# Autodistill Notebook
> Automated foundational model approach to segmenting and annotating dataset frames

This notebook is to be used when attmepting to use AutoDistill as a tool for automation of segmentation. Although novel and innovative, AutoDistill needs a lot of love and care to ensure that it works to the best of it's abiltities. Even with this, the accuracy of the correct class being specified is highly unlikely. If you use autodistill, please manually review each frame. 
<br>
<br>
AutoDistill is a basic foundational model that wraps Grounded-Segment-Anything, which is a project acting to combine GroundedDino and SAM (and it's variants).
<br> 
<br> 

### 1 - Installing required dependencies

In [ ]:
%pip install -q \
  autodistill \
  autodistill-grounded-sam \
  autodistill-yolov8 \
  roboflow \
  supervision==0.24.0

In [ ]:
import supervision as sv
import cv2
from autodistill.detection import CaptionOntology
from autodistill_grounded_sam import GroundedSAM
from pathlib import Path

import os
import shutil
import gc
import torch


### 2 - Loading and preparing frames for annotation
This module requires frames to be placed within the `data/autodistill_inputs` directory. <br>
More to come on how to extract frames into this directory. Assume there is images within this directory.
<br>
Otherwise, edit the `IMAGE_DIR_PATH` variable to reach frames.

TODO: MAKE NO HARDCODE

In [ ]:
IMAGE_DIR_PATH = "/home/aston/desk/uni-related/cos40005/cv4gt/data/autodistill_inputs"

image_paths = sv.list_files_with_extensions(
    directory=IMAGE_DIR_PATH,
    extensions=["png", "jpg", "jpg"])

print('image count:', len(image_paths))

In [ ]:
SAMPLE_SIZE = 16
SAMPLE_GRID_SIZE = (4, 4)
SAMPLE_PLOT_SIZE = (16, 10)

titles = [
    image_path.stem
    for image_path
    in image_paths[:SAMPLE_SIZE]]
images = [
    cv2.imread(str(image_path))
    for image_path
    in image_paths[:SAMPLE_SIZE]]

sv.plot_images_grid(images=images, titles=titles, grid_size=SAMPLE_GRID_SIZE, size=SAMPLE_PLOT_SIZE)

### 3 - Define Ontology
Our ontology is what defines what the models inside AutoDistill what to look for within the image. <br>
More to come. <br>
Ensure that the most amount of detail in describing objects is done, this is a very important step within the usage of AutoDistill.

In [ ]:
ontology_prompts = {
    "sideloader_arm": [
        "multi-jointed metallic sideloader arm of a garbage truck",
        "articulated hydraulic arm for lifting bins",
        "segmented robotic loading arm on a truck",
        "long mechanical arm attached to a garbage truck",
        "white mechanical arm with hydraulic components",
        "waste collection truck bin gripper",
        "automated bin collection mechanism",
        "weathered garbage truck sideloader mechanism",
        "mechanical bin lifter with hydraulic lines",
        "detached or exposed garbage truck grabber arm"
    ],
    "bin": [
        "garbage bin",
        "wheelie bin",
        "trash can",
        "rubbish bin",
        "waste container",
        "yellow-lidded recycling bin",
        "red-lidded refuse bin",
        "municipal waste collection bin",
        "curbside collection container",
        "Australian wheelie bin"
    ],
    "fallen_bin": [
        "toppled garbage bin",
        "fallen wheelie bin",
        "tipped-over trash can",
        "sideways recycling bin",
        "overturned waste container",
        "bin lying on its side",
        "horizontally positioned bin",
        "knocked-over rubbish bin",
        "collapsed bin position",
        "bin not in upright position"
    ],
    "person": [
        "person",
        "pedestrian",
        "man",
        "woman"
    ],
    "car": [
        "car",
        "vehicle",
        "automobile", 
        "sedan car",
        "SUV vehicle",
        "parked passenger vehicle",
        "driveway parked car"
    ],
    "moving_vehicle": [
        "moving car",
        "vehicle in motion",
        "approaching automobile",
        "driving car",
        "traffic vehicle",
        "vehicle traveling on road"
    ],
    "vegetation": [
        "tree",
        "shrub",
        "leafy tree",
        "bushy shrub",
        "bush",
        "residential lawn",
        "front yard grass",
        "manicured garden",
        "decorative landscaping"
    ],
    "mailbox": [
        "mailbox",
        "postbox",
        "letterbox",
        "residential mail post",
        "curbside mail container"
    ],
    "pole": [
        "stationary vertical pole",
        "slender metal street light pole",
        "thin signpost pole",
        "wooden utility pole",
        "simple upright pole"
    ],
    "residential_property": [
        "suburban home",
        "residential driveway",
        "house frontage",
        "residential property",
        "suburban front yard",
        "paved walkway",
        "residential curb",
        "concrete driveway"
    ],
    "bench": [
        "street bench",
        "public seating",
        "outdoor bench",
        "roadside resting spot",
        "park bench"
    ],
    "bicycle": [
        "bicycle",
        "bike",
        "push bike",
        "pedal cycle",
        "mountain bike",
        "leaning bicycle"
    ],
    "animals": [
        "dog",
        "cat",
        "domestic pet",
        "wild animal",
        "bird on ground",
        "possum",
        "kangaroo",
        "loose animal"
    ],
    "recreational_vehicle": [
        "parked caravan",
        "boat on trailer",
        "camper trailer",
        "motorhome",
        "towed recreational vehicle"
    ],
    "bollard": [
        "traffic bollard",
        "protective post",
        "short safety column",
        "roadside barrier",
        "concrete bollard"
    ],
    "discarded_items": [
        "discarded bottle",
        "abandoned container",
        "roadside litter",
        "glass debris",
        "plastic waste"
    ],
    "bus_shelter": [
        "bus stop shelter",
        "transit waiting area",
        "covered bus stop",
        "public transport shelter",
        "roadside waiting structure"
    ],
    "cyclist": [
        "person on bicycle",
        "bike rider",
        "cycling person",
        "moving cyclist",
        "bicycle operator"
    ],
    "hard_waste": [
        "discarded furniture",
        "dumped mattress",
        "abandoned appliance",
        "roadside electrical waste",
        "large unwanted items",
        "council pickup items",
        "illegal dumping",
        "unwanted household items",
        "bulk rubbish collection",
        "garbage bags outside bin"
    ],
    "motorbike": [
        "motorcycle",
        "scooter",
        "moped",
        "parked motorbike",
        "two-wheeled motor vehicle"
    ],
    "power_box": [
        "electrical utility box",
        "roadside power cabinet",
        "transformer box",
        "electrical distribution box",
        "green utility box"
    ],
    "shopping_cart": [
        "abandoned shopping trolley",
        "supermarket cart",
        "grocery trolley",
        "metal shopping cart",
        "retail cart"
    ],
    "sign": [
        "street sign",
        "traffic sign",
        "roadside information",
        "regulatory sign",
        "warning sign"
    ],
    "truck": [
        "large truck",
        "delivery truck",
        "commercial vehicle",
        "heavy vehicle",
        "utility truck"
    ],
    "van": [
        "passenger van",
        "delivery van",
        "minivan",
        "cargo van",
        "commercial van"
    ]
}

ontology=CaptionOntology(ontology_prompts)

### 4 - Run AutoDistill GroundedSegmentAnything

In [ ]:
DATASET_DIR_PATH = "/home/aston/desk/uni-related/cos40005/cv4gt/data/autodistill_outputs"


base_model = GroundedSAM(ontology=ontology)
dataset = base_model.label(
    input_folder=IMAGE_DIR_PATH,
    output_folder=DATASET_DIR_PATH)

### 5 - Preview generated annotations 

In [ ]:
ANNOTATIONS_DIRECTORY_PATH = "/data/autodistill_outputs/train/labels"
IMAGES_DIRECTORY_PATH = "/data/autodistill_outputs/train/labels/images"
DATA_YAML_PATH = "/data/autodistill_outputs/data.yaml"